# End-to-end action rules mining on Telco Customer Churn

A self-contained hands-on tour of the package:

1. Load the Telco Customer Churn CSV (`notebooks/data/telco.csv`, 7,043
   customers).
2. Define intrinsic and transition **utility tables** so the miner can
   report economic value alongside statistical uplift.
3. Fit `ActionRules` with calibrated support/confidence thresholds.
4. Compute analytic **confidence intervals** (auto-switching Wald/Wilson) for
   the realistic rule gain.
5. **Export** the rule set to JSON for downstream consumption.

Every choice mirrors the one-page workflow recommended in the package docs
and the second-article companion script `article2/code/end_to_end.py`. The
result file `outputs/telco_rules.json` is the canonical JSON payload — load
it back with `action_rules.input.Input` to recover an `Output` object
without rerunning the miner.


In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Locate repo root from the notebook's working directory.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Repository root with pyproject.toml not found.')

TELCO_CSV = REPO_ROOT / 'notebooks' / 'data' / 'telco.csv'
OUT_DIR = REPO_ROOT / 'notebooks' / 'telco_tour' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
print(f'Repo root: {REPO_ROOT}')
print(f'Telco CSV: {TELCO_CSV.relative_to(REPO_ROOT)}')
print(f'Out dir:   {OUT_DIR.relative_to(REPO_ROOT)}')


Repo root: C:\Users\LukasSykora\OneDrive - Ogilvy\Documents\Soukrome\action rules\repo\action-rules
Telco CSV: notebooks\data\telco.csv
Out dir:   notebooks\telco_tour\outputs


## 1. Load the Telco Customer Churn dataset

The CSV ships with the repository (semicolon-separated). `SeniorCitizen`
is encoded as `{0, 1}` so we cast it to string to keep the action-rules
encoder happy.


In [2]:
df = pd.read_csv(TELCO_CSV, sep=';')
df['SeniorCitizen'] = df['SeniorCitizen'].astype(str)
print(f'shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print('target distribution:')
print(df['Churn'].value_counts(normalize=True).round(3).to_string())
df.head()


shape: 7,043 rows x 21 columns
target distribution:
Churn
No     0.735
Yes    0.265


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Utility tables

Two tables drive the *realistic rule gain* computation:

- **Intrinsic utility** — value of being in a state. The headline entry is
  `('Churn', 'No') = 400` — keeping a customer is worth 400 currency units
  in expected lifetime value. The other entries record the (negative) cost
  of having the customer subscribed to a paid add-on service.
- **Transition utility** — cost or saving of *changing* an attribute value.
  Upsell costs (e.g. switching `OnlineSecurity` from `No` to `Yes`) are
  recorded as negatives.

These numbers are illustrative; in production they should come from the
business team's economic model.


In [3]:
INTRINSIC = {
    ('Churn', 'No'): 400.0, ('Churn', 'Yes'): 0.0,
    ('OnlineSecurity',   'Yes'): -10.0, ('OnlineSecurity',   'No'): 0.0,
    ('DeviceProtection', 'Yes'):  -8.0, ('DeviceProtection', 'No'): 0.0,
    ('TechSupport',      'Yes'): -12.0, ('TechSupport',      'No'): 0.0,
    ('StreamingTV',      'Yes'):  -5.0, ('StreamingTV',      'No'): 0.0,
}
TRANSITION = {
    ('OnlineSecurity',   'No', 'Yes'): -5.0,
    ('DeviceProtection', 'No', 'Yes'): -4.0,
    ('TechSupport',      'No', 'Yes'): -6.0,
    ('StreamingTV',      'No', 'Yes'): -3.0,
}


## 3. Mine action rules

Stable attributes describe *who the customer is* (gender, senior status,
partnership). Flexible attributes describe *what we can change* (service
level, contact channel, add-ons). The miner discovers rules of the form

> *given stable conditions*, **changing the flexible attribute X from value
> A to value B** is associated with **the target moving from undesired to
> desired state**.

Calibrated thresholds (`min_*_support = 220 / 110`, confidence 0.6) yield
~20 rules — enough variety to demonstrate categorisation without flooding
the notebook.


In [4]:
from action_rules import ActionRules

STABLE = ['gender', 'SeniorCitizen', 'Partner']
FLEXIBLE = [
    'PhoneService', 'InternetService', 'OnlineSecurity',
    'DeviceProtection', 'TechSupport', 'StreamingTV',
]
TARGET = 'Churn'
UNDESIRED, DESIRED = 'Yes', 'No'

ar = ActionRules(
    min_stable_attributes=2,
    min_flexible_attributes=1,
    min_undesired_support=220,
    min_desired_support=110,
    min_undesired_confidence=0.6,
    min_desired_confidence=0.6,
    intrinsic_utility_table=INTRINSIC,
    transition_utility_table=TRANSITION,
)
ar.fit(
    data=df,
    stable_attributes=STABLE,
    flexible_attributes=FLEXIBLE,
    target=TARGET,
    target_undesired_state=UNDESIRED,
    target_desired_state=DESIRED,
)
n_rules = len(ar.output.action_rules)
print(f'mined {n_rules} action rules')


mined 24 action rules


## 4. Inspect a few rules

The pretty-printed notation reads naturally; the AR notation is the
machine-friendly equivalent. Both expose the underlying rule fields
(support, confidence, uplift, realistic rule gain) inline.


In [5]:
for i, line in enumerate(ar.get_rules().get_pretty_ar_notation()[:3]):
    print(f'Rule {i}: {line}\n')


Rule 0: If attribute 'gender' is 'Female', attribute 'Partner' is 'No', attribute 'InternetService' value 'Fiber optic' is changed to 'DSL', attribute (flexible is used as stable) 'OnlineSecurity' is 'No', attribute (flexible is used as stable) 'DeviceProtection' is 'No', then 'Churn' value 'Yes' is changed to 'No with support: 161, confidence: 0.3996921269931663, uplift: 0.016483010613374986, support of undesired part: 279, confidence of undesired part: 0.6355353075170843, support of desired part: 161, confidence of desired part: 0.62890625, max_rule_gain: 400.0, realistic_rule_gain: 105.77662300683373, realistic_dataset_gain: 46435.93750000001.

Rule 1: If attribute 'gender' is 'Female', attribute 'Partner' is 'No', attribute 'InternetService' value 'Fiber optic' is changed to 'DSL', attribute 'OnlineSecurity' value 'No' is changed to 'Yes', attribute (flexible is used as stable) 'DeviceProtection' is 'No', then 'Churn' value 'Yes' is changed to 'No with support: 114, confidence: 0.5

## 5. Confidence intervals (analytic, auto Wald/Wilson)

`analytic_type='auto'` selects Wilson when the per-side group is small or
the proportion is extreme, and Wald otherwise. We also pass `threshold=150.0`
to categorise rules at the *break-even cost* of 150 currency units per
retained customer.


In [6]:
results = ar.confidence_intervals(
    data=df,
    method='analytic',
    analytic_type='auto',
    metric='realistic_rule_gain',
    threshold=150.0,
)

summary = pd.DataFrame([
    {
        'rule_index': r.rule_index,
        'support': r.support,
        'gain': round(r.realistic_rule_gain_point, 1),
        'gain_lo': round(r.realistic_rule_gain_ci_lower, 1),
        'gain_hi': round(r.realistic_rule_gain_ci_upper, 1),
        'category': r.category.value if r.category else None,
    }
    for r in results
]).sort_values('gain', ascending=False).reset_index(drop=True)
summary.head(10)


,rule_index,support,gain,gain_lo,gain_hi,category
0,17,392,221.5,198.7,244.3,accept
1,21,392,221.5,198.7,244.3,accept
2,2,439,212.5,190.3,234.6,accept
3,13,439,212.5,190.3,234.6,accept
4,8,458,208.0,186.1,230.0,accept
5,15,458,208.0,186.1,230.0,accept
6,16,552,203.2,182.4,223.9,accept
7,5,552,203.2,182.4,223.9,accept
8,19,548,202.1,183.5,220.6,accept
9,23,548,202.1,183.5,220.6,accept


## 6. Export the rule set to JSON

`get_export_notation()` returns a JSON string with every rule's components,
including the freshly attached CI fields. It is the canonical interchange
format — load it back via `action_rules.input.Input`.


In [7]:
json_path = OUT_DIR / 'telco_rules.json'
with open(json_path, 'w', encoding='utf-8') as fh:
    fh.write(ar.get_rules().get_export_notation())
size_kb = json_path.stat().st_size / 1024
print(f'wrote {json_path.relative_to(REPO_ROOT)} ({size_kb:.1f} KB)')


wrote notebooks\telco_tour\outputs\telco_rules.json (27.4 KB)


## 7. Round-trip the JSON through `Input`

Useful sanity check: parse the exported JSON back to an `Output` object and
verify the rule count survives.


In [8]:
from action_rules.input import Input

with open(json_path, 'r', encoding='utf-8') as fh:
    payload = fh.read()
loaded = Input().import_action_rules(payload)
print(f'round-tripped {len(loaded.action_rules)} rules from JSON')


round-tripped 24 rules from JSON


## Where next?

- Move on to `02_visual_diagnostics.ipynb` for figure-quality plots of these
  same rules: churn-rate panel by attribute, bootstrap distribution + analytic
  overlay, forest plot, and a bootstrap-vs-analytic CI-width scatter.
- Bump support/confidence thresholds up or down to see how many rules
  survive — the categorisation should be insensitive to small changes (see
  `notebooks/inference_studies/06_threshold_sensitivity.ipynb`).
- Replace the utility tables with your own economic model to score *your*
  rules in *your* currency.
